# Final PCR Prediction (Classification)

In [192]:
import pandas as pd
import joblib
import numpy as np

### Preprocessing

#### Before Preprocessing

In [193]:
dataset_x_all = pd.read_csv("../datasets/TestDatasetExample.csv")

#### Type-specific preprocessing

In [194]:
dataset_x_all = dataset_x_all.drop("ID",axis=1)
dataset_x_all['original_shape_VoxelVolume'] = dataset_x_all['original_shape_VoxelVolume'].astype(np.float64)
categorical_names = dataset_x_all.select_dtypes(np.int64).columns

continuous_names = dataset_x_all.select_dtypes(np.float64).columns

dataset_x_all = dataset_x_all.replace([999], np.nan)

#### General Preprocessing

In [195]:
Xs = dataset_x_all.copy()

lower_quartile = Xs[continuous_names].quantile(0.25)
upper_quartile = Xs[continuous_names].quantile(0.75)
iqr = upper_quartile - lower_quartile

lower_bound = lower_quartile - 1.5 * iqr
upper_bound = upper_quartile + 1.5 * iqr

Xs[continuous_names] = np.where(Xs[continuous_names] < lower_bound, lower_bound, Xs[continuous_names])
Xs[continuous_names] = np.where(Xs[continuous_names] > upper_bound, upper_bound, Xs[continuous_names])

In [196]:
categorical_ranges = {
    "pCR (outcome)": (0,1),
    "ER": (0, 1),
    "PgR": (0, 1),
    "HER2": (0, 1),
    "TrippleNegative": (0, 1),
    "ChemoGrade": (1, 3),
    "Proliferation": (1, 3),
    "HistologyType": (1, 2),
    "LNStatus": (0, 1),
    "TumourStage": (1, 4),
    "Gene": (0, 1)
}

In [197]:
data_train_cat = pd.concat([Xs[categorical_names]], axis=1)
for col in data_train_cat.columns:
    
    # Truncate data to be within range for categorical features
    data_train_cat[col] = np.where(data_train_cat[col] > categorical_ranges[col][1], categorical_ranges[col][1],  data_train_cat[col])
    data_train_cat[col] = np.where(data_train_cat[col] < categorical_ranges[col][0], categorical_ranges[col][0],  data_train_cat[col])

In [198]:
imputer_knn = joblib.load('../exports/pcr_imputer_knn.pkl')
z_scaler = joblib.load('../exports/pcr_z_scaler_knn.pkl')

Xs = pd.DataFrame(imputer_knn.transform(Xs), columns=Xs.columns, index=Xs.index)
Xs[continuous_names] = z_scaler.transform(Xs[continuous_names])

Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,0.504364,0.0,0.0,0.0,1.0,3.0,3.0,1.5,0.0,2.0,...,-1.371318,-1.394522,0.144652,-0.865572,-0.330322,0.120273,-0.693680,1.192940,1.330040,-0.674673
1,0.812058,0.0,0.0,1.0,0.0,2.0,1.0,1.0,0.0,3.0,...,-0.590947,-0.589272,0.334554,1.711255,-0.869227,-0.819336,0.294567,-0.075352,-0.412285,0.449077
2,0.637815,0.0,0.0,0.0,1.0,3.0,3.0,1.0,1.0,4.0,...,-0.466675,-0.461018,1.529057,-0.044204,-0.405955,0.929642,-0.804259,1.001595,0.897202,-0.803130


Notes: HistologyType 1.5????????????????????????????????????????????????????????????????????????? LMAO

### Feature Selection

### Prediction

In [ ]:
models = joblib.load('../exports/pcr_models.pkl')

for key in models.keys():
    print(f"{key}: {models[key].predict(Xs)}")

# Then choose a model ig
# And write to csv don't forget

Logistic Regression: [1. 1. 1.]
Naive Bayes (Gaussian): [1. 1. 1.]
KNN (k=3): [0. 1. 1.]
SVM Hard (linear, C=100): [1. 1. 1.]
SVM RBF: [1. 1. 1.]
Decision Tree: [0. 1. 0.]
Random Forest (n=10): [0. 1. 1.]
MLP (50,50): [1. 1. 1.]
LDA: [1. 1. 1.]
